# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library. All dataset entities such as record sets, fields, and columns are referenced explicitly by their `@id`, ensuring clarity and reproducibility following best FAIR data practices.

### Dataset Source
Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and explore records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
# Show metadata description and title
print("Title:", dataset.metadata.name)
print("Description:", dataset.metadata.description)


## 2. Data Overview
Review the available record sets and their fields. All lookups are by `@id`. This step helps identify the entrypoints (record sets) and fields within each as defined by Croissant.


In [ ]:
# List all available record sets by their `@id` and human-readable name.
print('Available record sets:')
record_sets = dataset.metadata.record_sets

for rs in record_sets:
    print(f"- @id: {rs.id} | name: {rs.name}")
    if hasattr(rs, 'fields') and rs.fields:
        print("   Fields:")
        for field in rs.fields:
            print(f"     - @id: {field.id} | name: {field.name}")
    print()

## 3. Data Extraction
Extract data from one or more key record sets identified above. Each record set is queried by its `@id`.


In [ ]:
# Example: Extract records from the main record set containing protein data.

# [EDIT] Select the correct record set by its `@id`.
# After reviewing the overview above, edit the record set ID below to point to the main quantitative record set.
# Here we use 'cr:RecordSet/quantitative-data', but replace with the actual one from the listing if different.

# Find all record set IDs for demonstration
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print('Record set IDs:', record_set_ids)

# For this FAIR2 dataset the protein abundance table is usually under a main quantitative record set,
# here we select the first record set as an example. Adjust as needed.
selected_record_set_id = record_set_ids[0]
print(f"Selected record set: {selected_record_set_id}")

# Load records to a DataFrame
records = list(dataset.records(record_set=selected_record_set_id))
df = pd.DataFrame(records)
print(f"Columns ({len(df.columns)}):", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group using selected fields, referencing all by their known `@id` (which correspond to column names from Croissant).

In [ ]:
# Example: Select a numeric field such as 'Molecular Weight' (MW) for analysis.

# [EDIT]: Use the correct Croissant field `@id` as shown in the Data Overview (e.g., 'cr:Field/molecular_weight')
# We'll try common protein table columns. Edit as needed per your schema.

# Look up all columns for reference
print(df.columns)

# Here, let's try 'MW' or 'Molecular_Weight' if present, else choose a suitable numeric field
if 'MW' in df.columns:
    numeric_field = 'MW'
elif 'Molecular_Weight' in df.columns:
    numeric_field = 'Molecular_Weight'
elif len([c for c in df.columns if 'weight' in c.lower()]) > 0:
    numeric_field = [c for c in df.columns if 'weight' in c.lower()][0]
else:
    numeric_field = df.select_dtypes(include='number').columns[0]  # fallback
print(f"Numeric field selected: {numeric_field}")

# Filter for values > threshold
threshold = 10000  # 10k Da for proteins
filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (
    pd.to_numeric(filtered_df[numeric_field], errors='coerce') - pd.to_numeric(filtered_df[numeric_field], errors='coerce').mean()
) / pd.to_numeric(filtered_df[numeric_field], errors='coerce').std()

print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field, e.g., protein accession/ID if present
possible_group_fields = [c for c in df.columns if ('accession' in c.lower() or 'description' in c.lower() or 'type' in c.lower())]
if possible_group_fields:
    group_field = possible_group_fields[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable group field present in DataFrame.")


## 5. Visualization
Let's visualize the distribution of the selected numeric field after filtering and normalization.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the histogram of the normalized numeric field.
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True)
plt.title(f'Normalized Distribution of {numeric_field} (> {threshold})')
plt.xlabel(f"{numeric_field} (normalized)")
plt.ylabel("Count")
plt.show()

# If group_field available, show a boxplot by group
if 'group_field' in locals():
    n_groups = filtered_df[group_field].nunique()
    n_to_show = min(n_groups, 10)
    plt.figure(figsize=(10,6))
    sns.boxplot(data=filtered_df[:100], x=group_field, y=numeric_field)
    plt.title(f'{numeric_field} by {group_field} (first {n_to_show} groups)')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
We successfully loaded and programmatically explored the FAIR^2 dataset using the `mlcroissant` library, referencing all data entities by their unique `@id` for full provenance and reproducibility. We performed initial exploratory steps, including visualization of a key numeric field, with all field selection and groupings transparently referenced via Croissant metadata conventions. Further analysis or modeling can be performed based on these clean, well-documented data extracts.